# Part B: deep learning for referable DR (Colab)

Runs the whole Part B pipeline on a Colab GPU: data check → small CNN from scratch → EfficientNet-B0 frozen vs fine-tuned → comparison on validation → Grad-CAM on the winner → optional hybrid → **one** test evaluation → runtime notes.

**Before running:** Runtime → Change runtime type → **GPU**. Set `DRIVE_PROJECT` below to the group-Drive folder that contains `cache_clahe_512/` (or `cache_clahe_512.zip`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime > Change runtime type > GPU"
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT = '/content/drive/MyDrive/aptos-referable-dr'   # <- edit: folder with cache_clahe_512 (folder or .zip)
REPO = 'https://github.com/youngseols/aptos-referable-dr.git'   # private repo: use a token URL or upload the repo
BRANCH = 'main'

In [ ]:
import os
if not os.path.exists('/content/aptos-referable-dr'):
    !git clone -q -b $BRANCH $REPO /content/aptos-referable-dr
%cd /content/aptos-referable-dr
!git log --oneline -1
!pip -q install -r requirements.txt scikit-learn pandas matplotlib

## Copy the image cache to the Colab disk
Reading thousands of PNGs from Drive inside the training loop is slow, so the cache is copied once to local disk (`data/cache_clahe_512`). Training then preloads all images into RAM.

In [ ]:
import time, os
t0 = time.time()
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/cache_clahe_512'):
    if os.path.exists(f'{DRIVE_PROJECT}/cache_clahe_512.zip'):
        !unzip -q "$DRIVE_PROJECT/cache_clahe_512.zip" -d data/
    else:
        !cp -r "$DRIVE_PROJECT/cache_clahe_512" data/
print(len(os.listdir('data/cache_clahe_512')), 'images copied in', round(time.time() - t0), 's')

## 5. Hardware (recorded for the report)

In [ ]:
!python part_b/hardware.py

## 1. Data loader and split check
Grades are converted to the referable label (grade ≥ 2) while keeping the original 0–4 grade; class counts, IDs, duplicate leakage and missing images are checked.

In [ ]:
!python part_b/check_data.py

## 2. Small CNN from scratch (deep-learning baseline)

In [ ]:
!python part_b/train.py --model smallcnn --lr 1e-3

## 3. Transfer learning: EfficientNet-B0, frozen backbone vs fine-tuning

In [ ]:
!python part_b/train.py --model efficientnet_b0 --mode frozen --lr 1e-3

In [ ]:
!python part_b/train.py --model efficientnet_b0 --mode finetune --lr 1e-3

Optional extra rows for the comparison (only if time allows): last stage only, or ResNet-50.

In [ ]:
# !python part_b/train.py --model efficientnet_b0 --mode partial --lr 1e-3
# !python part_b/train.py --model resnet50 --mode finetune --lr 1e-3 --batch-size 24

## Optional: hybrid (CNN features → PCA → classical classifier)

In [ ]:
!python part_b/hybrid.py --arch efficientnet_b0

## Compare on validation and choose the winner
The winner is the run with the highest **validation** AUROC. Part A's validation predictions are added to the ROC plot for the classical-vs-deep comparison.

In [ ]:
!python part_b/compare.py --extra results/part_a/preds_A_rf_all.csv
import pandas as pd
from IPython.display import Image, display
table = pd.read_csv('results/part_b/comparison_val.csv'); display(table)
WINNER = table[table.run.str.startswith('hybrid') == False].iloc[0].run   # Grad-CAM needs a CNN run
print('winner for Grad-CAM:', WINNER)
display(Image('results/part_b/roc_val.png')); display(Image('results/part_b/learning_curves.png'))

## 4. Grad-CAM on the winning model (validation examples)

In [ ]:
!python part_b/gradcam.py --run $WINNER --split val --n 3
display(Image(f'results/part_b/gradcam_{WINNER}_val.png'))

## Final test evaluation — run ONCE
Only after every choice above is fixed. Thresholds come from the validation set.

In [ ]:
RUNS = ' '.join(r for r in table.run if not r.startswith('hybrid'))
!python part_b/final_test.py --runs $RUNS
!python part_b/hybrid.py --arch efficientnet_b0 --final-test
!python part_b/gradcam.py --run $WINNER --split test --n 3
display(pd.read_csv('results/part_b/test_metrics.csv'))
display(Image(f'results/part_b/gradcam_{WINNER}_test.png'))

## Runtime notes and save results to Drive

In [ ]:
!python part_b/compare.py --extra results/part_a/preds_A_rf_all.csv
print(open('results/part_b/runtime_hardware.md').read())
!mkdir -p "$DRIVE_PROJECT/part_b_results" && cp -r results/part_b/. "$DRIVE_PROJECT/part_b_results/"
!mkdir -p "$DRIVE_PROJECT/part_b_checkpoints" && cp outputs/part_b/checkpoints/*.pt "$DRIVE_PROJECT/part_b_checkpoints/"
print('Commit results/part_b/ (small files) to the repository; checkpoints stay on Drive.')